# 📊 03. 데이터 통계 + 인사이트 리포트 — H&M 커머스 (강사용 예시)

> 🔧 **강사용 안내**: 통계 미션·AI 감사의 답안엔 **예시 풀이**가 들어 있습니다. **정답이 아니라 "이렇게도 쓸 수 있다"의 한 사례**입니다. 학생 답이 다른 변수·다른 검정을 골랐어도, 절차(가정점검→선택→해석)를 지켰다면 맞습니다.

이 노트북은 **01_데이터이해_전처리 → 02_데이터_EDA → 03_데이터_통계(이 노트북)** 3파트 파이프라인의 **마지막**입니다. 02 에서 관찰만 하고 넘긴 질문("이 차이가 우연인지")을 여기서 통계로 확인하고, 프로젝트를 **인사이트 리포트**로 마무리합니다.

**구성**
1. **정제본 불러오기** — 01 이 만든 표를 그대로 이어 씁니다. 
2. **가이드 — 통계 레시피 갤러리** — day08(추론통계)·day09(가설검정·회귀)에서 배운 도구를 **전부** 모아 뒀습니다. **막힐 때 찾아 쓰는 참고서**입니다.
3. **선택 통계 미션 카탈로그** — ⚠️ **통계는 필수가 아닙니다.** 발제문의 필수 목표는 EDA·시각화·인사이트입니다. 이 절은 "차이가 우연인지까지 따지고 싶은 사람"을 위한 **선택**이며, **2~3개만 골라도 충분**합니다.
4. **최종 인사이트 리포트** — 이 프로젝트의 마무리, **필수**입니다.
5. **AI 협업 검증** — 오류가 심어진 AI 초안을 실제로 재계산해 잡아내는 실습입니다.
6. **제출 체크리스트** — 발제문 필수 목표를 다 담았는지 스스로 확인합니다.

## 1. 정제본 불러오기
01 이 저장한 정제본(`output/hm_clean.csv`)을 읽습니다. **원본 3개 테이블은 다시 읽지 않습니다.**

In [ ]:
# [제공 코드] 이 노트북에서 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')
import os
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pingouin as pg
import scikit_posthocs as sp                                 # 비모수 사후검정(Dunn) — pingouin 에 없음
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.contingency_tables import Table        # 카이제곱 사후분석(조정된 잔차)

if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=KOREAN_FONT, rc={'axes.unicode_minus': False})

In [ ]:
# [제공 코드] 정제본 로드 — 있으면 그대로 쓰고, 없으면 표준 규칙으로 그 자리에서 재현합니다.
clean_path = 'output/hm_clean.csv'
# 이 노트북의 예시가 쓰는 컬럼 — 하나라도 없으면 예시를 돌릴 수 없어 표준 규칙으로 재현합니다.
NEEDED = ['t_dat', 'age', 'price', 'sales_channel_id', 'club_member_status']

df = None
if os.path.exists(clean_path):
    loaded = pd.read_csv(clean_path)
    lack = [c for c in NEEDED if c not in loaded.columns]
    if lack:
        print(f"정제본에 이 노트북의 예시가 쓰는 컬럼이 없습니다: {', '.join(lack)}")
        print('예시를 돌리기 위해 표준 규칙으로 재현합니다.')
        print('여러분의 정제본을 그대로 쓰려면 01 에서 이 컬럼들을 남긴 채 다시 저장하세요.')
    else:
        df = loaded
        print('01 의 정제본을 불러왔습니다:', df.shape)

        # 01 에서 남겨 두기로 한 값이 있으면 이 노트북의 예시를 위해서만 걸러내고,
        # 무엇을 왜 걸렀는지 알려 줍니다(여러분의 선택을 부정하는 것이 아닙니다).
        fixed = []
        bad_age = (df['age'] < 10) | (df['age'] > 99)
        if bad_age.any():
            fixed.append(f'비현실 나이 {bad_age.sum()} 행')
            df = df[~bad_age]
        bad_price = df['price'] <= 0
        if bad_price.any():
            fixed.append(f'0 이하 가격 {bad_price.sum()} 행')
            df = df[~bad_price]
        bad_ch = ~df['sales_channel_id'].isin([1, 2])
        if bad_ch.any():
            fixed.append(f'정의에 없는 채널 코드 {bad_ch.sum()} 행')
            df = df[~bad_ch]
        if df['club_member_status'].isna().any():
            fixed.append('멤버십 결측 -> Unknown')
            df['club_member_status'] = df['club_member_status'].fillna('Unknown')
        if fixed:
            print('아래 항목이 남아 있어, 이 노트북의 예시를 돌리기 위해 여기서만 걸러 썼습니다.')
            print('(01 에 기록한 여러분의 선택은 바뀌지 않습니다 — 이 df 에만 적용되는 임시 조치입니다.)')
            for item in fixed:
                print('  -', item)
            print('예시용:', df.shape)
            print('  - 이 규칙을 의도해서 고른 것이라면 그대로 진행하세요.')
            print('    (그 값들을 살려 분석하려면 아래 레시피의 필터를 여러분 규칙에 맞게 고쳐 쓰면 됩니다.)')
            print('  - 01 을 아직 안 끝냈거나 실수로 빠뜨린 것이라면, 01 을 완료한 뒤 다시 저장하세요.')

if df is None:
    print('표준 규칙으로 재현합니다 — 원칙은 01 을 먼저 끝내는 것입니다.')
    tr = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/transactions_hm.csv')
    cu = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/customer_hm.csv')
    ar = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/articles_hm.csv')
    df = tr.merge(cu, on='customer_id', how='inner').merge(ar, on='article_id', how='inner')
    df = df.drop_duplicates()
    df = df[(df['age'] >= 10) & (df['age'] <= 99)]      # 비현실 연령 제거
    df = df[df['price'] > 0]                             # 0원 거래 제외
    df = df[df['sales_channel_id'].isin([1, 2])]         # 채널 코드 정상값만
    df['club_member_status'] = df['club_member_status'].fillna('Unknown')
    print('표준 규칙 재현 후:', df.shape)

df['t_dat'] = pd.to_datetime(df['t_dat'])   # CSV 왕복하며 날짜형이 문자열로 풀렸으면 다시 변환

# 이 노트북에서 쓰는 파생변수 — 정제본에 이미 있어도 여기서 다시 만들어 둡니다.
df['month'] = df['t_dat'].dt.month
df['age_group'] = (df['age'] // 10 * 10).astype(int)

df_s = df.sample(n=3000, random_state=42)   # 무거운 계산·그림에 재사용할 표본
print('정제본 준비 완료:', df.shape)
display(df.head(3))

## 2. 가이드 — 통계 레시피 갤러리 (막힐 때 여기)
day08(추론통계)·day09(가설검정·회귀)에서 배운 도구를 **목적별로 전부** 모아 뒀습니다. ⚠️ **가이드에는 다 나오지만, 3절 미션은 이 중 2~3개만 실제로 해 보면 됩니다.** 아래 예시는 모두 위에서 로드한 정제본 `df` 로 실제 실행됩니다.

### 2-1. 모집단·표본·표준오차·신뢰구간 (day08)
**이 절에서 할 일**: 표본으로 모집단을 추정하는 도구(CLT·표준오차·신뢰구간)를 복습합니다.

이 정제본(113,218행)도 사실 **전체 H&M 고객이 아니라 Kaggle 이 공개한 표본**입니다 — 우리가 보는 모든 평균·비율은 **모수(참값)의 추정치**입니다. day08 은 이 추정이 얼마나 믿을 만한지 재는 도구들입니다.

In [ ]:
# [레시피] 중심극한정리(CLT) — 표본크기 n 이 커질수록 표본평균의 분포가 정규분포로 수렴한다
rng = np.random.default_rng(42)
pop = df['price'].values                      # price 자체는 오른쪽으로 치우친 분포
pop_std = pop.std(ddof=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, [1, 30, 100]):
    means = rng.choice(pop, size=(2000, n), replace=True).mean(axis=1)
    sns.histplot(means, bins=30, ax=ax)
    ax.set_title(f'n={n}  경험적SE={means.std():.4f}\n이론SE={pop_std/np.sqrt(n):.4f}')
plt.suptitle('CLT — n 이 커질수록 표본평균 분포가 좁고 대칭인 정규분포에 가까워진다')
plt.tight_layout()
plt.show()
print('price 자체는 치우쳤지만(오른쪽 꼬리), n=100 표본평균은 이미 종모양에 가깝다.')

In [ ]:
# [레시피] 표준오차 SE = σ/√n — stats.sem() 이 한 번에 계산해 준다
for n in [40, 160]:                            # 160 = 40 의 4배
    empirical_se = rng.choice(pop, size=(2000, n), replace=True).mean(axis=1).std()
    print(f'n={n:>4}: 경험적 SE={empirical_se:.5f}   이론 SE(σ/√n)={pop_std/np.sqrt(n):.5f}')
print('-> n 을 4배로 늘리면 SE 는 대략 √4=2 배 작아진다(더 안정적인 추정).')

In [ ]:
# [레시피] 점추정 vs 구간추정 — 신뢰구간(stats.t.interval)
sample = rng.choice(pop, 200, replace=False)
point_estimate = sample.mean()                  # 점추정 — 숫자 하나
ci95 = stats.t.interval(0.95, df=len(sample) - 1, loc=point_estimate, scale=stats.sem(sample))
pop_mean = pop.mean()   # 참값 -- 표본으로 낸 추정이 이 값에 얼마나 가까운지 견준다
print(f'점추정(표본평균) = {point_estimate:.4f}   95% 신뢰구간 = ({ci95[0]:.4f}, {ci95[1]:.4f})   '
      f'모평균(참값) = {pop_mean:.4f}')
print('참값이 구간 안에 있는가?', ci95[0] <= pop.mean() <= ci95[1])
print()
print('올바른 해석: "이 방법으로 100번 표본을 뽑아 구간을 만들면, 그중 약 95개가 모평균을 포함한다."')
print('흔한 오해: "이 구간에 모평균이 있을 확률이 95%" 라는 말은 정확하지 않다',
      '(모평균은 고정값이라 확률을 논할 대상이 아니다 — 확률은 "구간을 만드는 절차" 쪽에 붙는다).')
print()
print('신뢰구간으로 주장 판단: 예를 들어 누군가 "평균 단가는 0.05 다" 라고 주장하면,',
      '0.05 가 이 신뢰구간 밖에 있으므로 그 주장은 이 데이터와 맞지 않는다고 볼 수 있다.')

**p-value 직관**: p-value 는 "**귀무가설(H₀)이 참이라고 가정했을 때**, 지금 관측한 값(또는 더 극단적인 값)이 나올 확률"입니다. 작을수록 "H₀ 가 맞다면 이런 결과가 나오기 어렵다"는 뜻이라 H₀ 를 의심하게 됩니다. **흔한 오해**: p-value 는 ❌ "H₀ 가 참일 확률"이 아니고, ❌ "결과가 우연이 아닐 확률"도 아니며, ❌ 작을수록 무조건 "효과가 크다"는 뜻도 아닙니다(뒤에서 계속 나올 "유의≠실질"의 출발점입니다).

> ✅ **여기까지 되면 통과**: 표본이 커질수록 표준오차가 왜 작아지는지, 신뢰구간이 무엇을 보장하는지 말로 설명할 수 있으면 됩니다.

### 2-2. 가설검정 프레임 (day09)
**이 절에서 할 일**: 가설검정의 공통 언어(H₀/H₁·α·오류·양측/단측)를 정리합니다. 이후 모든 검정이 이 틀을 그대로 씁니다.

| 용어 | 뜻 |
| --- | --- |
| **귀무가설 H₀** | "차이(효과) 없다" — 기본 입장 |
| **대립가설 H₁** | "차이(효과) 있다" — 보이고 싶은 주장 |
| **유의수준 α** | 우연을 진짜라고 오해할 위험의 상한(보통 0.05) |
| **검정통계량** | t·F·χ² 등 — 데이터를 H₀ 아래에서 얼마나 벗어났는지로 요약한 값 |
| **p-value** | H₀ 가 참일 때 이 값(또는 더 극단)이 나올 확률. **p < α** 면 H₀ 기각 |
| **1종 오류** | H₀ 가 참인데 기각(잘못된 "차이 있다") — 확률 α |
| **2종 오류** | H₀ 가 거짓인데 기각 못함(못 알아챈 진짜 차이) — 확률 β |

**양측 vs 단측**: H₁ 이 "다르다(≠)"면 **양측**(기본값, `alternative='two-sided'`), "더 크다/작다"면 **단측**(`'greater'`/`'less'`)입니다. ⚠️ **방향을 반대로 잡으면 결론이 뒤집힙니다** — 같은 데이터에서 옳은 방향의 단측 p 는 양측 p 의 정확히 절반이지만, 틀린 방향으로 잡으면 p 가 1 에 가까워져 유의하지 않게 됩니다. **방향은 데이터를 보기 전에 정해야** 합니다(보고 나서 유리한 쪽으로 바꾸면 1종 오류가 부풀어 오릅니다). 이 노트북은 특별한 말이 없으면 **양측**을 씁니다.

> ✅ **여기까지 되면 통과**: H₀/H₁ 을 스스로 세울 수 있고, 단측을 잘못 골랐을 때 결론이 어떻게 뒤집히는지 설명할 수 있으면 됩니다.

### 2-3. 가정 점검 절차 + 검정 선택표
**이 절에서 할 일**: 평균을 비교하기 전에 **1) 정규성 → 2) 등분산**을 확인하고, 그 결과로 어떤 검정을 쓸지 고릅니다. 이후 모든 평균 비교 미션이 이 절차를 그대로 따릅니다.

In [ ]:
# [레시피] 1) 집단별 정규성 — pg.normality + Q-Q Plot (연령대별 단가로 예시)
display(pg.normality(data=df, dv='price', group='age_group'))
fig, ax = plt.subplots(figsize=(4.5, 4))
pg.qqplot(df[df['age_group'] == 30]['price'], dist='norm', ax=ax)
ax.set_title('30대 단가 Q-Q Plot (대표로 한 집단만)')
plt.tight_layout()
plt.show()
print('모든 연령대에서 정규성은 기각된다(price 가 오른쪽으로 치우쳐서). 다만 표본이 수천~수만 건이라',
      'CLT 로 평균 비교는 견딘다 -> 그렇다면 갈림길은 다음의 등분산이다.')

# [레시피] 2) 등분산 — pg.homoscedasticity(Levene). equal_var=False 면 등분산 가정이 깨진 것
lev = pg.homoscedasticity(data=df, dv='price', group='age_group')
display(lev)
print('등분산 equal_var =', bool(lev['equal_var'].iloc[0]))

**3) 검정 선택표** — 위 1·2번의 결과에 따라 아래 표로 검정을 고릅니다(대표본이면 정규성 위반은 CLT 로 어느 정도 버티지만, 등분산 위반은 검정 종류를 바꿉니다).

| 데이터 유형 | 등분산 만족 | 등분산 위반 | 정규성 심하게 위반 + 소표본 |
| --- | --- | --- | --- |
| 2집단 독립 | Student t-검정 | **Welch t-검정** | Mann-Whitney U (`pg.mwu`) |
| 2집단 대응(같은 대상) | 대응 t-검정 | (해당 없음) | Wilcoxon 부호순위 (`pg.wilcoxon`) |
| 3집단 이상 | ANOVA + Tukey | **Welch-ANOVA + Games-Howell** | Kruskal-Wallis + Dunn(`sp.posthoc_dunn`) |
| 범주 × 범주(연관) | 카이제곱 독립성(+Cramér's V, 조정잔차) | 〃 | 〃 |
| 범주 1개(비율 확인) | 카이제곱 적합도 | 〃 | 〃 |

> ✅ **여기까지 되면 통과**: 정규성·등분산 결과를 보고 이 표에서 검정을 정확히 짚어낼 수 있으면 됩니다.

### 2-4. 평균 비교 — t-검정 3종 + 효과크기
**이 절에서 할 일**: 상황별 t-검정 세 가지(1표본·대응·독립)를 실행하고, p-value 와 함께 **Cohen's d(효과크기)** 를 항상 같이 봅니다.

In [ ]:
# [레시피] 1) 1표본 t-검정 — '평균 단가가 0.025 라는 주장'이 맞는지
res_1samp = pg.ttest(df['price'], 0.025)
display(res_1samp)
print('실제 평균 =', round(df["price"].mean(), 4), ' T =', round(res_1samp["T"].iloc[0], 2),
      ' p =', res_1samp['p_val'].iloc[0], ' d =', round(res_1samp['cohen_d'].iloc[0], 3))

# [레시피] 2) 대응(paired) t-검정 — 같은 상품군의 온라인/오프라인 평균가를 짝지어 비교
piv = df.pivot_table(index='product_group_name', columns='sales_channel_id',
                      values='price', aggfunc='mean').dropna()
res_paired = pg.ttest(piv[2], piv[1], paired=True)   # 2=온라인, 1=오프라인, 상품군 12개가 짝
display(res_paired)
print('상품군', len(piv), '개 짝 중 온라인이 평균적으로 더 비싼가? p =', round(res_paired['p_val'].iloc[0], 4),
      ' d =', round(res_paired['cohen_d'].iloc[0], 3))

# [레시피] 3) 독립 2표본 t-검정 — 등분산이 깨졌으므로(2-3에서 미리 확인) Welch(correction=True)
online = df[df['sales_channel_id'] == 2]['price']
offline = df[df['sales_channel_id'] == 1]['price']
res_ind = pg.ttest(online, offline, correction=True)
display(res_ind)
print('온라인 평균 =', round(online.mean(), 4), ' 오프라인 평균 =', round(offline.mean(), 4),
      ' T =', round(res_ind['T'].iloc[0], 3), ' d =', round(res_ind['cohen_d'].iloc[0], 3))

> ✅ **여기까지 되면 통과**: 세 t-검정이 서로 다른 상황(주장 검증·짝지은 비교·두 독립 집단)에 쓰인다는 것을 구분하고, `correction=True` 가 왜 필요했는지(2-3 의 등분산 결과) 설명할 수 있으면 됩니다.

### 2-5. 모수 vs 비모수 — 정규성이 심하게 깨졌을 때
**이 절에서 할 일**: 표본이 작거나 정규성이 심하게 깨졌을 때 쓰는 **순위 기반 검정**을 t-검정과 짝지어 봅니다. (이 데이터는 표본이 커 t-검정도 견디지만, 방법 자체를 알아 둡니다.)

In [ ]:
# [레시피] Mann-Whitney U — 독립 2표본의 비모수 버전(무거우니 표본 2000개씩으로 데모)
online_s = online.sample(2000, random_state=42)
offline_s = offline.sample(2000, random_state=42)
mwu = pg.mwu(online_s, offline_s)
display(mwu)
print('RBC(방향 있는 효과크기, -1~1) =', round(mwu["RBC"].iloc[0], 3),
      ' CLES(한쪽이 더 클 확률) =', round(mwu["CLES"].iloc[0], 3))

# [레시피] Wilcoxon 부호순위 — 대응 t-검정의 비모수 버전(위 상품군 12쌍 재사용)
wil = pg.wilcoxon(piv[2], piv[1])
display(wil)
print('p =', round(wil["p_val"].iloc[0], 5), ' -> 대응 t-검정(2-4 2)과 결론이 같은가?')

> ✅ **여기까지 되면 통과**: `RBC`·`CLES` 가 무엇을 뜻하는지 말할 수 있고, 같은 데이터에서 모수 검정과 비모수 검정의 결론이 (대개) 일치한다는 것을 확인했으면 됩니다.

### 2-6. 3집단 이상 비교 — ANOVA/Welch-ANOVA + 사후검정, 비모수 Kruskal+Dunn
**이 절에서 할 일**: 02 에서 넘어온 질문 — **"연령대별 평균 단가 차이가 통계적으로 유의한가?"** — 를 정식 검정으로 확인합니다. 2-3 에서 이미 **정규성 위반 + 등분산 위반**(Levene p=1.2e-34)을 확인했으므로 **Welch-ANOVA** 가 적절합니다.

In [ ]:
# [레시피] 일반 ANOVA(η² 포함) vs 등분산이 깨졌을 때의 Welch-ANOVA — 나란히 비교
aov = pg.anova(data=df, dv='price', between='age_group', detailed=True)
welch_aov = pg.welch_anova(data=df, dv='price', between='age_group')
print('일반 ANOVA   F =', round(aov["F"].iloc[0], 3), ' p =', aov['p_unc'].iloc[0],
      ' eta^2 =', round(aov['np2'].iloc[0], 4))
print('Welch-ANOVA  F =', round(welch_aov["F"].iloc[0], 3), ' p =', welch_aov['p_unc'].iloc[0],
      ' eta^2 =', round(welch_aov['np2'].iloc[0], 4), '  <- 등분산이 깨졌으므로 이쪽을 채택')

# [레시피] 사후검정 — Tukey(등분산 가정) vs Games-Howell(등분산 미가정, 이번엔 이쪽이 맞다)
tukey = pg.pairwise_tukey(data=df, dv='price', between='age_group')
gh = pg.pairwise_gameshowell(data=df, dv='price', between='age_group')
print('Tukey 유의한 쌍:', int((tukey['p_tukey'] < 0.05).sum()), '/', len(tukey))
print('Games-Howell 유의한 쌍:', int((gh['pval'] < 0.05).sum()), '/', len(gh), '(등분산 미가정 -> 채택)')

# [레시피] 비모수 대안 — Kruskal-Wallis + Dunn (scikit-posthocs, pingouin 에 없음)
kw = pg.kruskal(data=df, dv='price', between='age_group')
display(kw)
dunn = sp.posthoc_dunn(df, val_col='price', group_col='age_group', p_adjust='bonferroni')
iu = np.triu_indices_from(dunn.values, k=1)
print('Dunn 유의한 쌍:', int((dunn.values[iu] < 0.05).sum()), '/', len(iu[0]),
      '  -> Games-Howell 과 거의 같은 결론(경로가 달라도 수렴)')

**결론**: F=55.4(Welch), p≈0 로 연령대별 단가 차이는 **유의**하지만 η²=0.0037(0.4%)로 **실질적 효과는 매우 작습니다** — 02 의 질문에 대한 답은 "유의하지만 미미하다"입니다. 사후검정(Games-Howell)·비모수(Dunn) 모두 36쌍 중 약 절반이 유의해 **비슷한 결론에 수렴**합니다.

> ✅ **여기까지 되면 통과**: 등분산이 깨졌을 때 왜 Welch-ANOVA·Games-Howell 을 쓰는지 설명하고, η² 가 작다는 것과 p 가 유의하다는 것이 왜 모순이 아닌지 말할 수 있으면 됩니다.

### 2-7. 범주형 관계 — 카이제곱
**이 절에서 할 일**: **적합도**(한 범주형 변수가 특정 비율을 따르는지)와 **독립성**(두 범주형 변수가 관련 있는지)을 구분해 씁니다. 02 에서 넘어온 두 번째 질문 — **"연령대와 채널 선호가 독립적인가?"** — 도 여기서 답합니다.

In [ ]:
# [레시피] 1) 적합도 검정 — '온·오프라인 비중이 50:50 일 것이다' 라는 가설 확인
obs_counts = df['sales_channel_id'].value_counts().sort_index().values
exp_counts = np.array([obs_counts.sum() / 2, obs_counts.sum() / 2])   # H0: 50:50
chi2_gof, p_gof = stats.chisquare(obs_counts, exp_counts)
print('관측', obs_counts, ' 기대(50:50)', exp_counts, ' chi2 =', round(chi2_gof, 1), ' p =', p_gof)
print('-> p 가 사실상 0 이라 50:50 이라는 가정은 기각(온라인이 훨씬 많다).')

In [ ]:
# [레시피] 2) 독립성 검정 — 연령대 x 채널 (02 의 두 번째 질문)
# 먼저 원본 9개 연령대로 시도 -> 기대빈도 가정(5 이상) 점검부터
expected_raw, observed_raw, _ = pg.chi2_independence(data=df, x='age_group', y='sales_channel_id')
print('원본 9개 연령대: 최소 기대빈도 =', round(expected_raw.values.min(), 2), '(<5 이면 가정 위배)')

# 90대(6건)·80대(60건)처럼 표본이 아주 작은 연령대가 원인 -> 70대 이상을 '70+' 로 묶어 재시도
df['age_group2'] = df['age_group'].clip(upper=70)
expected, observed, chi_stats = pg.chi2_independence(data=df, x='age_group2', y='sales_channel_id')
pearson = chi_stats[chi_stats['test'] == 'pearson'].iloc[0]
print('그룹핑 후 최소 기대빈도 =', round(expected.values.min(), 1), '(5 이상 -> 가정 만족)')
print('chi2 =', round(pearson['chi2'], 2), ' p =', pearson['pval'], ' Cramer V =', round(pearson['cramer'], 4))

resid_df = pd.DataFrame(Table(observed.values).standardized_resids,
                        index=observed.index, columns=observed.columns)
display(resid_df.round(2))
print('-> 30대(온라인 쪽 +20.5) 가 기대보다 가장 두드러지게 온라인을 많이 쓴다.')

**결론**: 연령대와 채널은 **독립이 아니다**(χ²=704.5, p≈0)지만 Cramér's V=0.079 로 **연관은 약합니다** — 02 의 두 번째 질문에 대한 답도 "유의하지만 약하다"입니다. 그리고 **가정 점검(기대빈도 ≥5)이 실전에서 왜 중요한지**도 함께 보였습니다 — 원본 연령대 그대로는 최소 기대빈도가 1.85 로 가정이 깨져, 90대·80대를 '70+' 로 묶어야 검정을 믿을 수 있었습니다.

> ✅ **여기까지 되면 통과**: 적합도와 독립성의 차이를 설명하고, 기대빈도가 5 미만일 때 왜 결과를 그대로 믿으면 안 되는지 말할 수 있으면 됩니다.

### 2-8. 효과크기 기준표 — "유의"와 "실질"은 다르다
**이 절에서 할 일**: p-value 는 "우연이 아니다"만 말합니다. **차이가 얼마나 큰지**는 아래 효과크기로 따로 봅니다. 표본이 클수록(이 데이터처럼 11만 건) 아주 작은 차이도 p 가 유의해지므로, **효과크기 없이 p 만 보는 것은 위험**합니다.

| 효과크기 | 작음 | 중간 | 큼 | 쓰이는 곳 |
| --- | --- | --- | --- | --- |
| Cohen's d | 0.2 | 0.5 | 0.8 | t-검정 |
| η²(에타제곱) | 0.01 | 0.06 | 0.14 | ANOVA |
| Cramér's V | 0.1 | 0.3 | 0.5 | 카이제곱(2x2 기준, 자유도 클수록 기준 낮아짐) |
| RBC(비모수) | 0.1 | 0.3 | 0.5 | Mann-Whitney U |

![효과 크기 해석 가이드](../../day10_데이터분석_종합실습/images/효과크기_해석.png)

> ✅ **여기까지 되면 통과**: 2-4~2-7 에서 나온 효과크기 숫자들을 이 표에 놓고 "작다/중간/크다"를 스스로 분류할 수 있으면 됩니다.

### 2-9. 회귀 — 상관에서 다중회귀·LINE 진단까지
**이 절에서 할 일**: "무엇이 고객의 총구매액을 예측하는가"를 상관 → 단순회귀 → 다중회귀 → 범주형 더미까지 확장하고, 그 회귀를 믿어도 되는지 **LINE 4가정**으로 진단합니다.

In [ ]:
# [레시피] 상관 -> 단순회귀(r^2) -> 다중회귀(Adj R^2 비교)
cust = df.groupby('customer_id').agg(
    total=('price', 'sum'), n=('price', 'count'), age=('age', 'first'),
    online_ratio=('sales_channel_id', lambda s: (s == 2).mean()),
).reset_index()
print('나이-총구매액 상관 r =', round(cust['age'].corr(cust['total']), 4))

m_simple = smf.ols('total ~ age', data=cust).fit()
m_multi = smf.ols('total ~ age + n + online_ratio', data=cust).fit()
# 두 모형을 같은 형식으로 찍어 세로로 견준다(이름은 10칸 왼쪽 정렬)
for label, m in [('단순(나이)', m_simple), ('다중', m_multi)]:
    print(f'{label:<10} R^2={m.rsquared:.4f}  Adj R^2={m.rsquared_adj:.4f}')
print('-> 변수를 추가하면 R^2 는 늘 오르지만, 진짜 쓸모는 Adj R^2(변수 개수로 벌점)로 비교한다.',
      '여기선 구매빈도 n 이 설명력 대부분을 끌어올렸다.')

In [ ]:
# [레시피] 범주형 변수를 더미(원-핫)로 — 계수는 '기준 범주와의 차이'로 읽는다
cust_club = df.groupby('customer_id').agg(
    total=('price', 'sum'), club=('club_member_status', 'first'),
).reset_index()
m_cat = smf.ols('total ~ C(club)', data=cust_club).fit()   # C() 가 더미(k-1개)를 자동 생성
print(m_cat.params.round(5))
print('-> 알파벳/등장 순 첫 범주(ACTIVE)가 기준이 되어 빠지고, 나머지 계수는 ACTIVE 대비 차이다.')

In [ ]:
# [레시피] LINE 4가정 잔차 진단 — 이 회귀(다중모형)를 믿어도 되는가
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))
ax1.scatter(m_multi.fittedvalues, m_multi.resid, alpha=0.1, s=5)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_title('(L)inearity/(E)qual variance — 잔차 vs 적합값')
pg.qqplot(m_multi.resid, dist='norm', ax=ax2)
ax2.set_title('(N)ormality — 잔차 Q-Q Plot')
plt.tight_layout()
plt.show()

dw = durbin_watson(m_multi.resid)
bp_stat, bp_p, _, _ = het_breuschpagan(m_multi.resid, m_multi.model.exog)
print(f'(I)ndependence Durbin-Watson={dw:.3f}(2 근처면 OK)   '
      f'(E) Breusch-Pagan p={bp_p:.3g}(작으면 등분산 위배)')

**외삽(extrapolation) 위험**: 이 회귀선은 **관측된 나이·구매빈도 범위 안에서만** 믿을 수 있습니다. 데이터에 없는 극단값(예: 200세, 구매 1000회)을 넣어 예측하면 회귀식은 무너집니다. **유의≠실질**: 위 계수들의 p 는 유의하지만 단순회귀의 R²=0.0026 은 "나이가 총구매액의 0.26%만 설명한다"는 뜻입니다 — 표본이 커서(93,053명) 아주 작은 관계도 유의해진 것이지, 관계가 크다는 뜻이 아닙니다.

> ✅ **여기까지 되면 통과**: Adj R² 가 왜 필요한지, 더미의 '기준 범주'가 무엇인지, LINE 네 글자가 각각 무엇을 뜻하는지 설명할 수 있으면 됩니다.

## 3. 선택 통계 미션 카탈로그
⚠️ **다시 한 번 — 통계는 필수가 아닙니다.** 발제문의 필수 목표는 EDA·시각화·인사이트입니다. 아래는 **2절 가이드를 직접 손으로 해 보는 연습**이며, **2~3개만 골라도 충분**합니다.

| # | 미션 | 난이도 | 도구 | 예상 시간 |
| --- | --- | --- | --- | --- |
| A | 채널별 단가 차이 검정 | ★★ | `pg.homoscedasticity`·`pg.ttest` | 15분 |
| B | 연령대별 단가 차이 + 사후검정 *(02 연계)* | ★★★ | `pg.anova`/`welch_anova`·사후검정 | 20분 |
| C | 연령대 × 채널 독립성 검정 *(02 연계)* | ★★★ | `pg.chi2_independence`·조정잔차 | 20분 |
| D | 멤버십별 단가 차이 검정 | ★★★ | `pg.anova`·`pg.pairwise_tukey` | 20분 |
| E | 고객 총구매액 회귀 + LINE 진단 | ★★★ | `smf.ols`·잔차 진단 | 25분 |

### 미션 A — 채널별 단가 차이 검정
온라인·오프라인의 평균 단가가 정말 다른지 **가정 점검 → 검정 선택 → 실행**의 전 과정을 **직접** 작성해 보세요(가이드 2-4 3번과 같은 질문이지만, 이번엔 스스로).

> ✅ **이런 결과면 잘 된 것**: 등분산 점검 결과에 맞는 t-검정을 고르면 T 값이 약 64.9 근처로 나옵니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검)·2-4(t-검정)

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
lev = pg.homoscedasticity(data=df, dv='price', group='sales_channel_id')
equal_var = bool(lev['equal_var'].iloc[0])
print('등분산 =', equal_var)

online = df[df['sales_channel_id'] == 2]['price']
offline = df[df['sales_channel_id'] == 1]['price']
res = pg.ttest(online, offline, correction=not equal_var)
display(res)
print('T =', round(res["T"].iloc[0], 3), ' p =', res['p_val'].iloc[0],
      ' d =', round(res['cohen_d'].iloc[0], 3))

> 🔧 **강사 Note**: 등분산이 깨져 있어(Levene 유의) Welch 를 골라야 T=64.897 이 나옵니다. 등분산 가정 t-검정을 잘못 고르면 T 값이 살짝 달라집니다(약 58대).

### 미션 B — 연령대별 단가 차이 + 사후검정
02 의 질문 — "연령대별 평균 단가 차이가 통계적으로 유의한가?" — 를 **직접** 검정하세요. 정규성·등분산을 점검하고, 그 결과에 맞는 ANOVA(또는 Welch-ANOVA)와 사후검정을 실행하세요.

> ✅ **이런 결과면 잘 된 것**: F 값이 약 55(Welch) 근처, η² 는 0.01 미만(매우 작음)으로 나오고, 사후검정에서 36쌍 중 절반가량이 유의하게 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검 선택표)·2-6(ANOVA/Welch-ANOVA)

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
lev = pg.homoscedasticity(data=df, dv='price', group='age_group')
equal_var = bool(lev['equal_var'].iloc[0])
if equal_var:
    aov = pg.anova(data=df, dv='price', between='age_group', detailed=True)
    posthoc = pg.pairwise_tukey(data=df, dv='price', between='age_group')
    p_col = 'p_tukey'
else:
    aov = pg.welch_anova(data=df, dv='price', between='age_group')
    posthoc = pg.pairwise_gameshowell(data=df, dv='price', between='age_group')
    p_col = 'pval'
print('등분산 =', equal_var, ' F =', round(aov['F'].iloc[0], 3), ' eta^2 =', round(aov['np2'].iloc[0], 4))
print('사후검정 유의한 쌍:', int((posthoc[p_col] < 0.05).sum()), '/', len(posthoc))

> 🔧 **강사 Note**: 등분산이 깨져 있어 Welch-ANOVA + Games-Howell 이 정답 경로입니다. F=52.8(일반)과 F=55.4(Welch)는 비슷하지만, 엄밀히는 후자를 보고합니다.

### 미션 C — 연령대 × 채널 독립성 검정
02 의 두 번째 질문 — "연령대와 채널 선호가 독립적인가?" — 를 카이제곱 독립성 검정으로 확인하세요. **기대빈도가 5 이상인지** 먼저 확인하고, 부족하면 연령대를 묶어 다시 검정하세요. Cramér's V 와 조정된 잔차까지 확인하세요.

> ✅ **이런 결과면 잘 된 것**: 그룹핑 후 최소 기대빈도가 5 이상이 되고, χ² 는 유의(p≈0)하지만 Cramér's V 는 0.1 미만(약함)으로 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-7(카이제곱)

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
df['age_group2'] = df['age_group'].clip(upper=70)   # 70대 이상을 '70+' 로 묶어 표본 확보
expected, observed, chi_stats = pg.chi2_independence(data=df, x='age_group2', y='sales_channel_id')
pearson = chi_stats[chi_stats['test'] == 'pearson'].iloc[0]
print('최소 기대빈도 =', round(expected.values.min(), 1))
print('chi2 =', round(pearson['chi2'], 2), ' p =', pearson['pval'], ' V =', round(pearson['cramer'], 4))
resid_df = pd.DataFrame(Table(observed.values).standardized_resids,
                        index=observed.index, columns=observed.columns)
display(resid_df.round(2))

> 🔧 **강사 Note**: 원본 9개 연령대 그대로 검정하면 90대(6건) 때문에 최소 기대빈도가 5 미만이 됩니다 — 그룹핑은 '숫자 맞추기'가 아니라 가정을 지키기 위한 것입니다.

### 미션 D — 멤버십별 단가 차이 검정
채널·연령대와는 다른 축인 **멤버십 상태(`club_member_status`)** 별로 단가가 다른지 검정하세요. 이번엔 등분산이 만족되는 경우인지 직접 확인해 보세요(가이드 2-6 은 연령대만 다뤘습니다).

> ✅ **이런 결과면 잘 된 것**: 등분산이 만족되어 일반 ANOVA를 쓰면 F 는 약 9 근처, p 는 유의하지만 η² 는 0.001 미만(극히 작음)으로 나오고, 사후검정에서는 딱 한 쌍만 유의하게 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-3(가정 점검)·2-6(ANOVA)

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
lev = pg.homoscedasticity(data=df, dv='price', group='club_member_status')
print('등분산 =', bool(lev['equal_var'].iloc[0]))                          # 이번엔 True 가 나온다
aov = pg.anova(data=df, dv='price', between='club_member_status', detailed=True)
print('F =', round(aov['F'].iloc[0], 3), ' p =', aov['p_unc'].iloc[0], ' eta^2 =', round(aov['np2'].iloc[0], 6))
tukey = pg.pairwise_tukey(data=df, dv='price', between='club_member_status')
sig = tukey[tukey['p_tukey'] < 0.05]
display(sig[['A', 'B', 'diff', 'p_tukey', 'hedges']])

> 🔧 **강사 Note**: F=8.913·p=0.000007 로 유의하지만 η²=0.000236 로 사실상 무의미한 크기입니다. Tukey 6쌍 중 ACTIVE-PRE-CREATE 단 한 쌍만 유의(p=0.00002)합니다 — '유의하다'가 '전부 다르다'는 뜻이 아니라는 좋은 사례입니다.

### 미션 E — 고객 총구매액 회귀 + LINE 진단
고객별 총구매액을 나이·구매빈도·온라인비중으로 설명하는 **다중회귀**를 적합하고, LINE 4가정을 잔차로 진단하세요(가이드 2-9 의 지침대로 만든 `cust`·`m_multi` 를 그대로 이어 써도 됩니다).

> ✅ **이런 결과면 잘 된 것**: R² 는 약 0.38, 세 계수 모두 p<0.001 로 유의하고, Durbin-Watson 은 2 근처(독립성 OK)지만 Breusch-Pagan p 는 사실상 0(등분산 위배)으로 나오면 됩니다.
> 🔧 **막히면 볼 곳**: 가이드 2-9(회귀·LINE 진단)

**예시 풀이 — 이렇게도 풀 수 있다는 한 사례입니다**

In [ ]:
# 가이드 2-9 에서 만든 cust·m_multi 를 그대로 재사용합니다(새로 만들어도 결과는 같습니다).
print(m_multi.summary())
dw = durbin_watson(m_multi.resid)
bp_stat, bp_p, _, _ = het_breuschpagan(m_multi.resid, m_multi.model.exog)
print('R^2 =', round(m_multi.rsquared, 4), ' Durbin-Watson =', round(dw, 3), ' Breusch-Pagan p =', bp_p)
print('-> 방향은 믿을 만하지만(계수 유의), 등분산이 깨져 있어 표준오차·p-value 의 정밀한 숫자는',
      '그대로 믿기 어렵다. 실무에서는 log(total) 로 다시 적합하거나 강건표준오차를 쓴다.')

> 🔧 **강사 Note**: 등분산·정규성이 깨진 상태에서도 계수의 '방향'(나이·구매빈도·온라인비중이 총구매액과 양의 관계)은 신뢰할 만하지만, 신뢰구간·p-value 의 소수점까지는 과신하지 않습니다.

## 4. 최종 인사이트 리포트 (필수)
여기부터는 **선택이 아닙니다.** 01→02→03 에서 본 것을 하나의 리포트로 묶는, 이 프로젝트의 마무리입니다.

**리포트 뼈대 — 다섯 조각**
1. **무엇을 물었나** — 이 분석의 질문·가설(00 에서 세운 비즈니스 목표와 연결).
2. **어떻게 봤나** — 데이터·처리 규칙 요약(무엇을 지우고 무엇을 남겼는지, 그 이유).
3. **무엇을 발견했나** — 수치 + 그래프로 뒷받침되는 핵심 발견 2~4개(통계를 했다면 p·효과크기도).
4. **그래서 무엇을 하나** — 발견을 실행 가능한 제안으로(누구에게·무엇을·어떻게).
5. **한계** — 이 결론을 과신하면 안 되는 이유.

**숫자를 말로 옮기는 법** — 3) 발견을 쓸 때는 항상 이 순서를 따르세요.
1. **숫자 재진술** — 무엇을 비교했고 값이 얼마였는지 있는 그대로 적는다.
2. **유의성 + 효과크기** — p-value(우연인지)와 Cohen's d·η²·Cramér's V(크기)를 함께 적는다.
3. **방향** — 그래서 무엇이 더/덜 한지 한 문장으로 정리한다.

이때 두 가지를 절대 헷갈리지 마세요 — **유의 ≠ 실질**(p 가 작다고 차이가 큰 것은 아님, 특히 표본이 클 때)과 **상관 ≠ 인과**(같이 움직인다고 한쪽이 다른 쪽의 원인은 아님)입니다.

**좋은 예 / 나쁜 예**

| | 나쁜 예 | 좋은 예 |
| --- | --- | --- |
| 발견 | "온라인이 잘 팔린다" | "온라인 평균 단가가 오프라인보다 **약 1.3배**(0.0299 vs 0.0228) 높고, 이 차이는 통계적으로 유의하며 효과크기(Cohen's d=0.376)도 작은~중간 수준이다." |
| 제안 | "마케팅을 더 하자" | "단가가 높은 온라인 채널에 매출 상위 상품군(상의류) 노출을 강화하고, 가입 대기(PRE-CREATE) 고객 대상 온라인 온보딩을 설계한다." |
| 한계 | (언급 없음) | "이 데이터는 **Kaggle 이 공개한 표본**이라 전체 H&M 고객이 아니고, **관찰 자료라 인과가 아니며**, `price` 는 실제 화폐가 아닌 **정규화된 상대값**이다." |

**한계·해석 규율 — 어떤 결론을 쓰든 빠뜨리면 안 되는 여섯 가지**
- **매출의 정의(수량이 없다)**: 거래 표에 **수량(quantity) 컬럼이 없습니다.** 그래서 `price` 를 더한 값은 엄밀히 "매출액"이 아니라 **"구매 단가의 총합"** 입니다 — 리포트에 이 말을 그대로 적어 두세요. 게다가 이 값은 실제 화폐가 아니라 **0~1 사이로 정규화된 상대값**입니다(직접 `df['price'].min()`·`max()`·`mean()` 으로 확인해 보세요). "1,000원"처럼 절대 금액으로 읽거나 통화 단위를 붙여 쓰면 안 됩니다. 참고로 발제문에는 통화 단위를 언급하라고 적혀 있지만 이 데이터의 `price` 는 그 서술과 맞지 않습니다 — **문서와 데이터가 다를 때는 데이터를 믿고, 그 차이를 리포트에 적으세요.**
- **표본 편향**: 이 데이터는 Kaggle 이 공개한 **일부 표본**입니다 — 특히 **온라인(`sales_channel_id=2`) 기록이 상대적으로 많이 담겨** 있어 채널 비중이 실제보다 온라인 쪽으로 왜곡돼 있을 수 있습니다. 그래서 채널·집단을 비교할 때는 **건수 그대로가 아니라 비율(%)이나 고객당 평균 구매액(ARPU)으로 보정**해 비교하세요.
- **상관 ≠ 인과**: 연령대·채널·멤버십과 단가 사이의 관계는 원인이 아니라 **관찰된 결과**입니다. "20대 단가가 높다 → 20대를 집중 타깃으로" 같은 도약을 하지 마세요(무작위 배정 실험이 아니므로 "온라인이라서 비싸졌다"도 말할 수 없습니다). 제안은 "이 세그먼트에서 **검증해 볼 가치가 있다**" 수준으로 씁니다.
  · 참고로 발제문은 이 대목에서 **성별**을 예로 들며 "20대 여성" 을 언급하지만, 이 데이터의 고객 테이블에는 **성별 컬럼이 아예 없습니다**(`customer_id`·`FN`·`Active`·`club_member_status`·`fashion_news_frequency`·`age` 6개). 발제문에 적힌 변수가 실제로 있는지 **직접 확인하고** 없으면 "이 표본에는 성별 정보가 없어 분석하지 못했다"고 한계에 적으세요.
- **시즌성**: 특정 월의 급증을 **"성장"으로 단정하지 마세요.** 블랙프라이데이·연말연시 같은 이벤트가 반영돼 있을 수 있습니다. 추세를 말하려면 여러 기간을 비교하거나 롤링 평균처럼 노이즈를 줄인 뒤 말하세요.
- **인기 ≠ 트렌드 선도**: 상품군·색상의 매출 상위는 **공급량·재고 정책·프로모션**의 결과일 수 있습니다. "상의류가 1위"는 사실이지만 "상의류가 트렌드를 이끈다"는 이 데이터로 말할 수 없습니다.
- **인플레이션·환율 미반영**: 시점별 물가·환율 차이가 보정되지 않았으므로 **절대적 추세 해석은 제한적**입니다 — 같은 시점 안에서의 **상대 비교**에 초점을 두세요.

**리포트 템플릿** — 아래 빈칸을 채우세요.

**모범 리포트 — 이렇게도 쓸 수 있다는 한 사례입니다**

**1) 무엇을 물었나** — 어떤 고객군·채널·상품군이 매출을 주도하는지, 그리고 그 차이가 우연이 아닌지 확인하고자 했다.

**2) 어떻게 봤나** — 거래·고객·상품 3개 표를 inner join 으로 결합(113,218행)하고, 중복·비현실 연령·0원 거래·잘못된 채널 코드를 제거했다. 멤버십 결측은 삭제 대신 `Unknown` 범주로 남겼다.

**3) 무엇을 발견했나** — 온라인 평균 단가가 오프라인보다 약 1.3배 높고 통계적으로 유의하다(Welch t=64.9, p≈0, d=0.376 — 작은~중간 효과). 연령대별·멤버십별 단가 차이도 유의하지만 효과크기는 매우 작다(η²<0.01). 연령대와 채널은 독립이 아니지만(χ²=704.5, p≈0) 연관은 약하다(V=0.079) — 다만 30대는 기대보다 뚜렷하게 온라인을 선호한다. 고객별 총구매액은 나이·구매빈도·온라인비중으로 약 38% 설명되지만(R²=0.382), 잔차 진단에서 등분산·정규성이 깨져 있어 정밀한 예측보다는 방향 참고용이다.

**4) 그래서 무엇을 하나** — 단가가 높은 온라인 채널에서 매출 상위 상품군(상의류)의 노출·추천을 강화한다. 30대처럼 온라인 선호가 뚜렷한 세그먼트에 맞춤 프로모션을 검토한다. 다만 효과크기가 작은 차이(연령대·멤버십)에 큰 자원을 쏟는 것은 우선순위가 낮다.

**5) 한계** — 이 데이터는 Kaggle 표본(전체 H&M 고객 아님)이고 온라인 기록이 상대적으로 많아 채널 비중이 왜곡될 수 있어 비율·ARPU 로 보정해 비교했다. 관찰 자료라 상관을 인과로 단정할 수 없고, 수량 컬럼이 없어 `price` 합은 "구매 단가 총합"이며 정규화된 상대값이라 절대 금액으로 해석할 수 없다. 월별 급증은 시즌성(연말·프로모션) 가능성이 있어 성장으로 단정하지 않았고, 물가·환율이 보정되지 않아 절대 추세보다 상대 비교에 한정했다. 회귀의 등분산·정규성 위반도 함께 고려해야 한다.

## 5. AI 협업 검증
리포트 초안 작성에 AI(챗봇형 도구)를 쓰면 빠르지만, AI 는 **숫자·인과·연관 강도를 그럴듯하게 지어내거나 과장할 수 있습니다(환각)**. 여기서는 실제 API 호출 없이, **이미 오류가 심어진 AI 초안을 직접 재계산해 잡아내는 연습**을 합니다.

**1) 좋은 프롬프트 4요소** — 맥락(무슨 데이터·무슨 분석인지) + 수치(내가 가진 정확한 값) + 요청(무엇을 써 달라는지) + 형식(불릿·길이 등), 그리고 "**준 수치 외의 값은 지어내지 말 것**"을 마지막에 덧붙이면 환각을 줄일 수 있습니다.

**2) AI 초안 감사 — 아래 초안에는 오류 3개가 숨어 있습니다**

> H&M 거래 데이터를 분석한 결과, **온라인 채널의 매출 비중이 약 90%에 달합니다.** 이는 온라인이 이미 압도적인 주력 채널임을 보여줍니다. 그렇다면 **오프라인 고객을 온라인으로 전환시키면 매출이 곧바로 증가할 것입니다.** 한편 상품군별로 보면 상의류(Garment Upper body)가 매출 1위인데, 이는 **20대가 상의를 집중적으로 구매하기 때문**입니다.

**요구사항** — 아래 답안 셀에서 세 주장을 각각 **코드로 재계산**해 대조하고, 대조표를 완성한 뒤 초안을 고쳐 쓰세요. 자가채점이 값을 확인하므로 **변수 이름은 아래에 적힌 그대로** 쓰고, 세 값 모두 **0~1 사이의 비율**로 두세요(백분율로 바꾸지 마세요 — 출력할 때만 100을 곱하면 됩니다).

| # | 재계산할 것 | 변수명 | 쓸 컬럼·값 |
| --- | --- | --- | --- |
| 1 | 온라인 채널의 **실제 매출 비중**(거래 건수가 아니라 **수익**=`price` 합 기준) | `online_share` | `sales_channel_id` 별 `price` 합에서 채널 `2` 의 몫 |
| 2 | 채널별 **가입 대기(PRE-CREATE) 회원 비중** | `offline_precreate` · `online_precreate` | `sales_channel_id` × `club_member_status` 교차표를 **행 기준 비율**로(`pd.crosstab(..., normalize='index')`) 만든 뒤 `'PRE-CREATE'` 열에서 채널 `1`·`2` 값 |
| 3 | 상의 구매 중 20대 비중 vs 전체 거래 중 20대 비중 | `share_upper_20s` · `share_all_20s` | `product_group_name == 'Garment Upper body'` 로 걸러낸 뒤 `age_group == 20` 의 비율, 그리고 전체 `df` 에서 같은 비율 |

숫자를 구한 뒤 **각 주장을 판정**하세요 — 1) 실제 매출 비중은 초안의 "약 90%"와 얼마나 다른가? 2) 두 채널의 PRE-CREATE 비중이 서로 크게 다르다면 그것은 "채널이 무작위로 배정되지 않았다"는 신호인데, 그렇다면 "전환하면 매출이 는다"는 **인과**를 이 데이터로 말할 수 있는가? 3) 상의 구매 중 20대 비중은 전체보다 높은가, 낮은가? 그 결과가 "20대가 집중 구매해서"라는 설명을 뒷받침하는가?

> ⚠️ **여기만 예외적으로 자가채점이 있습니다** — 재계산 자체가 맞는지 확인하기 위해서입니다(분석·서술은 여전히 채점하지 않습니다). 허용오차: **비중은 ±0.5%p**, 나머지는 **부등호 비교**입니다.

In [ ]:
# 1) 온라인 채널의 실제 매출 비중 (수익 기준 — 거래 건수 기준과 다릅니다)
rev_by_channel = df.groupby('sales_channel_id')['price'].sum()
online_share = rev_by_channel[2] / rev_by_channel.sum()
print('온라인 매출 비중 =', round(online_share * 100, 1), '%  (AI 초안 주장: 약 90%)')

# 2) 채널별 가입 대기(PRE-CREATE) 회원 비중 — '전환하면 매출 증가' 인과 주장 검증용
ct = pd.crosstab(df['sales_channel_id'], df['club_member_status'], normalize='index')
offline_precreate = ct.loc[1, 'PRE-CREATE']
online_precreate = ct.loc[2, 'PRE-CREATE']
print(f'PRE-CREATE 비중 — 오프라인={offline_precreate * 100:.2f}%  온라인={online_precreate * 100:.2f}%')

# 3) 상의 구매 중 20대 비중 vs 전체 고객(거래) 중 20대 비중
upper = df[df['product_group_name'] == 'Garment Upper body']
share_upper_20s = (upper['age_group'] == 20).mean()
share_all_20s = (df['age_group'] == 20).mean()
print(f'상의 구매 중 20대={share_upper_20s * 100:.1f}%   전체 거래 중 20대={share_all_20s * 100:.1f}%')

In [ ]:
# [자가채점] 재계산이 맞는지만 확인합니다(서술은 채점하지 않습니다).
assert abs(online_share * 100 - 74.7) < 0.5, '온라인 매출 비중 재계산을 다시 확인하세요'
assert offline_precreate < online_precreate, ('PRE-CREATE 는 오프라인보다 온라인에 훨씬 몰려 있어야 '
                                              '합니다(채널이 무작위 배정이 아니라는 근거)')
assert share_upper_20s < share_all_20s, ('상의 구매자 중 20대 비중이 전체보다 낮아야 합니다 '
                                         '("20대 집중구매" 주장이 근거 없음을 보여주는 지점)')
print('✅ 재계산 확인 완료')

**대조표(모범)**

| AI 초안 주장 | 재계산 값 | 판정 | 고친 문장 |
| --- | --- | --- | --- |
| "온라인 매출 비중이 약 90%" | 74.7% | ❌ 숫자 환각 | "온라인 매출 비중은 약 74.7%다" |
| "전환시키면 매출이 곧바로 증가" | PRE-CREATE 비중 오프라인 0.01% vs 온라인 2.80% — 채널은 무작위 배정이 아님 | ❌ 인과 오류 | "채널과 매출은 함께 관찰된 경향일 뿐이며, 전환이 매출 증가를 보장한다고 말하려면 A/B 테스트가 필요하다" |
| "20대가 집중 구매해서 상의가 1위" | 상의 구매 중 20대 39.7% < 전체 중 20대 41.0%(오히려 낮음) | ❌ 근거 없는 단정 | "상의가 매출 1위(38.2%)인 것은 사실이지만, 20대의 집중 구매가 그 이유라는 근거는 없다" |

**3) 반영 원칙** — AI 초안의 **모든 수치를 내 코드 출력과 한 줄씩 대조**하고, 인과·설명을 **과장하는 표현**(~하면 곧바로·~때문에 등)은 관찰 데이터에 맞게 완화합니다. **AI 는 표현을 빠르게 다듬는 도우미이고, 숫자·인과의 최종 책임은 사람에게 있습니다.**

## 6. 제출 체크리스트
제출 전, 발제문의 **필수 목표**를 다 담았는지 스스로 확인하세요(체크만 하는 절이라 자가채점은 없습니다).

| # | 항목 | 어디서 |
| --- | --- | --- |
| 1 | 비즈니스 목표 세우기 | `01_데이터이해_전처리` 1장 |
| 2 | 데이터 소스 설명 | `01_데이터이해_전처리` 2장 |
| 3 | [EDA] 규모 파악 | `01_데이터이해_전처리` |
| 4 | [EDA] 타입과 기술통계 | `01_데이터이해_전처리` 2장 + `02_데이터_EDA` |
| 5 | [전처리] 테이블 결합 | `01_데이터이해_전처리` |
| 6 | [전처리] 규칙 수립·실행(이유 포함) | `01_데이터이해_전처리` |
| 7 | [분석·시각화] 기준 컬럼 비교 + 3개 이상 그림 | `02_데이터_EDA` |
| 8 | [인사이트] 최소 1개 이상, 수치·그래프·해석 | `02_데이터_EDA` + 이 노트북 4절 |

통계(3절)를 했다면 **덤**입니다 — 위 8개 항목에는 포함되지 않지만 리포트를 더 탄탄하게 만듭니다.